# NSL-KDD Dataset: Feature Guide for Intrusion Detection

## label (Target Variable)

* **What it is:** The category of the network connection (normal or a specific attack type).  
* **Why useful:** This is what the model learns to predict — the final decision.

**Model guidance:**
- Target variable.
- For binary classification: convert all attack types to "attack".
- For multi-class: keep all categories.

## difficulty_level

* **What it is:** A score (0-21) indicating how hard this record is to classify correctly.  
* **Why useful:** Helps identify ambiguous or sophisticated attacks that might be frequently misclassified. Useful for analyzing model weaknesses.  
* **Model guidance:** **Remove before training.** It's a meta-feature about the dataset itself, not a characteristic of the live network traffic you'll be predicting on.

## Basic Connection Features

### Protocol Type (TCP, UDP, ICMP)

Indicates what communication method a network connection uses.

1. **TCP (Transmission Control Protocol):**  
   TCP is a connection-oriented protocol. This means both computers establish a reliable connection before data is exchanged.  
   - Connection-based and reliable  
   - Ensures all data arrives correctly and in order  
   - Used for web browsing, email, file transfers

2. **UDP (User Datagram Protocol)**  
   UDP is a connectionless protocol. It sends packets without checking whether they arrive. Fast: No handshake or acknowledgment.  
   - Connectionless and fast  
   - Does not guarantee delivery or order  
   - Used for streaming, gaming, VoIP

3. **ICMP (Internet Control Message Protocol)**  
   ICMP is not for sending data between users—it's mainly for network diagnostics and error reporting. Used by routers and devices to communicate problems or status.  
   - Used for network diagnostics and error messages  
   - Not for sending user data  
   - Used by tools like ping and traceroute (checking connectivity)

### service

In networking, service refers to the specific application or function running over a port (on top of a protocol like TCP or UDP). It tells you what type of communication is happening.

**Common examples:**
- HTTP – web browsing (website access)  
- FTP – file transfer between computers  
- SSH – secure remote login to a machine  
- SMTP – sending email  
- DNS – domain name lookup  
- POP3/IMAP – receiving email  
- HTTPS – secure web browsing  
- Telnet – remote login (insecure)  
- SNMP – network device management

- The protocol (TCP/UDP) handles how data is sent.  
- The service tells you what the data is for.

**Service Table:**

| Service | Purpose              | Typical Port |
|---------|----------------------|--------------|
| HTTP    | Web browsing         | 80 (TCP)     |
| SSH     | Secure remote login  | 22 (TCP)     |
| FTP     | File transfer        | 21 (TCP)     |
| SMTP    | Sending emails       | 25 (TCP)     |

When a computer connects to another device’s port 80, it is assumed to be requesting the HTTP (web) service.

### flag

Indicates the status/outcome of a network connection after it ends.

**Examples:**
- SF – connection completed normally  
- S0 – connection attempt but no reply  
- REJ – connection rejected  
- RSTO/RSTR – reset by origin/receiver  

This helps detect whether a connection behaved normally or suspiciously.

### src_bytes

Number of bytes sent from the source (client) to the destination during the connection.  
Shows how much data the attacker/victim is sending.

### dst_bytes

Number of bytes sent from the destination (server) back to the source.  
Helps compare traffic flow.

**Example:**
- src_bytes high, dst_bytes low → possible data upload  
- src_bytes low, dst_bytes high → downloading

## ⚠️ Traffic Pattern Features (The “Behavior” Features)

*These features summarize the pattern of traffic to/from the same host or service over a short window (usually 2 seconds).*

### serror_rate

Proportion of connections to the same destination that ended with a SYN error (connection setup failure).  
This usually indicates:
- port scanning  
- connection attempts that fail  
- possible SYN flood attack

### rerror_rate

Proportion of connections to the destination that ended with a REJ (reject) error.  
High values indicate:
- many attempts to closed ports  
- reconnaissance/scanning activity

### same_srv_rate

Percentage of the host’s connections that were made to the same service (same port type).

**Example:**  
If 80% of connections are to HTTP → same_srv_rate = 0.8

Helps detect patterns such as:
- repetitive access to one service  
- automated scanning tools

### diff_srv_rate

Percentage of connections to different services on the same host.

**Example:**  
Connecting to HTTP, then FTP, then SSH quickly → high diff_srv_rate

Indicates:
- port scans  
- multi-target probing

## 🔍 Content & Host-Based Features

*These features look "inside" the traffic or at longer-term host behavior, searching for signals of exploits or compromised systems.*

### num_root

Number of “root” (admin-level) shell accesses or privilege escalations in the session.  
Used to detect:
- privilege escalation attacks  
- compromised accounts

### num_file_creations

How many times files were created during the connection.  
Often associated with:
- malware dropping files  
- installing backdoors  
- FTP upload activities

### num_shells

Number of shell prompts (e.g., command-line sessions) opened.  
Useful for spotting:
- unauthorized remote access  
- attackers opening command shells  
- interactive sessions

### dst_host_count

Number of connections made to the same destination host within a certain time window.

High values may mean:
- DoS attack  
- repeated login attempts  
- heavy traffic to one server

### dst_host_srv_count

Number of connections to the same service on the destination host in the time window.

**Example:**  
50 connections to port 80 on the same server → high value.

Indicates:
- web scanning  
- focused attacks on one service

### dst_host_same_src_port_rate

Percentage of connections to the destination host that use the same source port.

High value might mean:
- automated tools  
- repeated probes using same port  
- abnormal traffic patterns

### dst_host_diff_srv_rate

Fraction of connections to the destination host that use different services (different ports).

High value indicates:
- port scanning  
- probing many services on the same host

## 🔵 A. BASIC FEATURES

These describe **basic properties of a single network connection**.

#### 1. duration

The **length of the connection** in seconds.  
Used to detect:
- long abnormal sessions  
- short repetitive attacks (scans, DoS bursts)

#### 2. land

Indicates whether the **source and destination IP address and port are the same**.

- **1 = land attack (same source/destination)**  
- **0 = normal**

Land attacks send packets from and to the same host to confuse systems.

#### 3. wrong_fragment

Number of **wrong (invalid or broken) IP fragments** in the packet.

High values indicate:
- fragmentation attacks  
- attempts to evade firewalls/IDS  
- malformed packets

#### 4. urgent

Number of packets with the **URG (urgent) flag** set in TCP.  
Normally rare. High values signal:
- stealth attacks  
- misuse of the TCP urgent feature  
- tunneling or abnormal communications

## 🔵 B. CONTENT FEATURES

These describe **what happens inside the connection**, including authentication and system access behavior.

#### 5. hot

A count of “hot indicators” such as:
- system calls  
- suspicious commands  
- accessing sensitive files  
- performing risky actions

Higher = suspicious activity inside the session.

#### 6. num_failed_logins

Number of **failed login attempts**.  
Indicates:
- password guessing  
- brute-force login attacks  
- unauthorized access attempts

#### 7. logged_in

Indicates if the user **successfully logged in**.

- **1 = login success**  
- **0 = login failure**  

Used to detect login-based attacks.

#### 8. num_compromised

Number of **compromised (corrupted) conditions** detected during this session.  
These include:
- repeated suspicious commands  
- known exploit patterns  
- unauthorized accesses

High values → strong sign of intrusion.

#### 9. root_shell

Indicates if the connection resulted in a **root (administrator) shell**.

- **1 = attacker got root shell**  
- **0 = no root access**  

Very strong indicator of a successful intrusion.

#### 10. su_attempted

Indicates attempts to perform the **“su” command** (switch user to root).

Values:
- **0 = no attempt**  
- **1 = “su root” attempted but failed**  
- **2 = successful attempt**

Shows attempts to escalate privileges.

#### 11. num_access_files

Number of attempts to access **sensitive system files** (e.g., /etc/passwd).  
High values → possible reconnaissance or exploitation.

#### 12. num_outbound_cmds

Number of **outbound FTP commands**.  
Normally 0 in most traffic.  
High = suspicious FTP activity or data exfiltration.

#### 13. is_host_login

Indicates if the login was a **“hotlist” login** from a known host account.

- 1 = host-based login  
- 0 = normal user login  

Used to detect compromised system accounts.

#### 14. is_guest_login

Indicates whether the login was a **guest account**.  
Guest accounts = low privileges, often abused by attackers.

Values:
- **1 = guest login**  
- **0 = non-guest login**

## 🔵 C. TIME-BASED TRAFFIC FEATURES (2-Second Window)

#### 15. count

Number of connections to the **same destination host** in the last 2 seconds.  
High = host is being targeted heavily → possible DoS or scan.

#### 16. srv_count

Number of connections to the **same service** on the destination host in last 2 seconds.  
E.g., repeated connections to port 80 → possible HTTP flood.

#### 17. srv_serror_rate

Percentage of connections to the **same service** in the last 2 seconds that had **SYN errors**.  
High values = same-service scanning or SYN flood.

#### 18. srv_rerror_rate

Percentage of connections to the **same service** in the last 2 seconds that resulted in **REJ errors**.  
High = attacker hitting closed ports on the same service.

#### 19. srv_diff_host_rate

Fraction of connections to the **same service but different hosts** in the last 2 seconds.

**Example:**  
- Attacker scans HTTP on many IPs → high srv_diff_host_rate  

Indicates horizontal scanning.

## 🔵 D. HOST-BASED TRAFFIC FEATURES (100-Connection Window)

#### 20. dst_host_same_srv_rate

Percentage of the last 100 connections to this host that used the **same service**.

High:
- normal web traffic  
- OR targeted attack on one service

#### 21. dst_host_srv_diff_host_rate

Percentage of connections using the **same service** but coming from **different hosts**.

**Example:**
- Many systems contacting the same web server → normal  
- But high + abnormal errors → distributed scan / DDoS

#### 22. dst_host_serror_rate

Percentage of the last 100 connections to the host that had **SYN errors**.  
High = likely targeted with failed TCP handshakes (SYN flood).

#### 23. dst_host_srv_serror_rate

Percentage of **same-service** connections that had **SYN errors**.  
High = SYN flood focused on a specific service (e.g., HTTP only).

#### 24. dst_host_rerror_rate

Percentage of connections to this host that resulted in **REJ errors**.  
High = attacker probing closed ports.

#### 25. dst_host_srv_rerror_rate

Percentage of **same-service** connections ending in **REJ errors**.  
High = focused scanning of one service’s closed ports.

### Overview of NSL-KDD Attack Types

The NSL-KDD dataset is an enhanced version of the original KDD Cup 99 dataset, designed for evaluating intrusion detection systems (IDS). It addresses issues like redundant records and class imbalance in the original. Each network connection in the dataset is labeled as either "normal" or one of 39 specific attack types. These attacks are grouped into four main categories: **Denial of Service (DoS)**, **Probe (or Surveillance)**, **User to Root (U2R)**, and **Remote to Local (R2L)**. The dataset includes 23 attack types in the training set and 17 additional ones in the testing set (some novel attacks not seen in training, to test generalization).

These categories represent different intrusion strategies, and models trained on NSL-KDD often classify traffic into these groups or treat them as multi-class problems. Below, I'll explain each category, its purpose, and a list of common specific attacks under it. The lists are based on standard breakdowns from the dataset documentation and research analyses.

#### 1. **Denial of Service (DoS)**
   - **Description**: These attacks aim to overwhelm a target system or network with excessive traffic or resource demands, making it unavailable to legitimate users. They often involve flooding the victim with packets to exhaust bandwidth, CPU, or memory. DoS is the most prevalent category in NSL-KDD, comprising about 50-60% of attack instances.
   - **Specific Attacks**:

| Attack Name     | Brief Explanation |
|-----------------|-------------------|
| back           | Sends TCP SYN-ACK packets to open ports, causing resource exhaustion. |
| land           | Spoofs SYN packets with the same source and destination IP/port (rare). |
| neptune        | SYN flood variant using ICMP echo requests to amplify traffic. |
| pod            | Ping-of-Death: Oversized ICMP packets to crash systems. |
| smurf          | ICMP broadcast amplification using directed broadcasts. |
| teardrop       | IP fragmentation overlap exploits to crash reassembly. |
| apache2        | Floods Apache servers with GET requests. |
| mailbomb       | Floods email servers with messages. |
| processtable   | Exhausts process tables via fork bombs. |
| udpstorm       | UDP packet storm to ports 1 and 2. |

#### 2. **Probe (Surveillance or Reconnaissance)**
   - **Description**: These are exploratory attacks focused on gathering information about a network, such as active hosts, open ports, or services running. They often precede other attacks (e.g., DoS or exploitation) and mimic legitimate scanning tools. Probes make up a smaller portion (~10%) of the dataset but are key for detecting early reconnaissance.
   - **Specific Attacks**:

| Attack Name   | Brief Explanation |
|---------------|-------------------|
| ipsweep      | Sweeps IP ranges to identify live hosts. |
| mscan        | Port scanning with Mail Security Scanner. |
| nmap         | Network mapper tool for host/port discovery. |
| portsweep    | Sweeps multiple ports on a single host. |
| saint        | Vulnerability scanner (similar to SATAN). |
| satan        | System Administration Tool for Analyzing Networks (port scanning). |

#### 3. **User to Root (U2R)**
   - **Description**: These attacks start with a valid user account (local or remote) and exploit software vulnerabilities to escalate privileges to superuser (root) level. They are stealthy and less common (~0.1-1% of instances), representing advanced persistent threats.
   - **Specific Attacks**:

| Attack Name      | Brief Explanation |
|------------------|-------------------|
| buffer_overflow | Exploits buffer overflow in a program to execute arbitrary code. |
| loadmodule      | Loads a kernel module to gain root access. |
| perl            | Uses Perl script to exploit a vulnerability. |
| rootkit         | Installs a rootkit to hide and maintain root access. |
| xterm           | Exploits X server display to execute commands as root. |

#### 4. **Remote to Local (R2L)**
   - **Description**: An attacker with no prior access sends packets to exploit vulnerabilities, guessing credentials, or using social engineering to gain local user access from a remote machine. These are common in real-world scenarios like phishing or weak authentication (~10-20% of instances).
   - **Specific Attacks**:

| Attack Name     | Brief Explanation |
|-----------------|-------------------|
| ftp_write      | Writes files via anonymous FTP. |
| guess_passwd   | Brute-force password guessing (e.g., via Telnet). |
| imap           | Exploits IMAP buffer overflow. |
| multihop       | Uses a stolen account to hop through multiple machines. |
| phf            | CGI script exploit on web servers. |
| spy            | Installs a sniffer on compromised hosts. |
| warezclient    | Downloads illegal software (illegal knowledge exploit). |
| warezmaster    | Distributes illegal software. |
| sendmail       | Exploits Sendmail vulnerabilities. |
| named          | DNS server exploit. |
| snmpgetattack  | SNMP community string guessing. |
| snmpguess      | SNMP brute-force. |
| xlock          | X Lock screen exploit. |
| xsnoop         | X Snooper for password sniffing. |



# **Network Intrusion Detection Dataset – Feature Report with Analogies**

---

## **1️⃣ Connection & Duration Features**

| Feature         | What It Captures                                              | Intuitive Analogy                                            | Normal Behaviour                                                                                                                                                               | Attacker Behaviour                                                                                                                                                                                                                               |
| --------------- | ------------------------------------------------------------- | ------------------------------------------------------------ | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| `duration`      | Total connection duration in seconds                          | How long a visitor stays inside the building                 | Short to moderate duration for completing a specific task (e.g., loading a webpage, transferring a file).                                                                      | Extremely short (e.g., fraction of a second in a DoS flood) or abnormally long (e.g., persistent backdoor connection).                                                                                                                           |
| `long_duration` | Flags unusually long connections                              | Visitor loiters for an unusually long time                   | Typically not set for most standard service connections.                                                                                                                       | Set for connections that are sustained for hours/days, indicating persistent threats like data exfiltration, C2 channels, or slow-scanning probes.                                                                                               |
| `time_slot`     | Categorizes connections into four 6-hour intervals            | Visitor enters in early morning, morning, afternoon, evening | Connections follow general human activity patterns, peaking during business hours and dropping at night.                                                                       | May show activity concentrated in off-hours (e.g., midnight to 6 AM) to avoid detection, or constant probing across all time slots.                                                                                                              |
| `is_peak_hour`  | Flags connections during busy periods (e.g., 9–11 AM, 2–5 PM) | Visitor arrives during rush hours                            | Most connections occur during peak hours, correlating with legitimate user activity.                                                                                           | May also occur during peak hours to blend in with normal traffic (e.g., phishing attempts) or may deliberately avoid them to target quieter systems with fewer defenders.                                                                        |

---

## **2️⃣ Protocol & Service Features**

| Feature                   | What It Captures                              | Intuitive Analogy                                                | Normal Behaviour                                                                                                   | Attacker Behaviour                                                                                                                                                                                                                                                                      |
| ------------------------- | --------------------------------------------- | ---------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------ | --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `protocol_type`           | Network protocol (TCP, UDP, ICMP)             | Type of delivery method used by the visitor (car, bike, walking) | Predominantly TCP for reliable web, email, file transfers. UDP for DNS, streaming. ICMP for diagnostics (ping).    | May abuse protocols: excessive ICMP for reconnaissance/smurf attacks; UDP floods for amplification attacks; unusual TCP flag combinations for stealth scanning or SYN floods.                                                                                                           |
| `service`                 | Service accessed (HTTP, FTP, SSH, etc.)       | Specific department visitor heads to (Accounting, HR, IT)        | Accesses common, expected services (HTTP on port 80, HTTPS on 443, SSH on 22 for management).                     | Targets unusual ports (e.g., SSH on non-standard port for stealth); attempts connection to vulnerable/legacy services (e.g., Telnet, SNMP); or tries to access administrative services (e.g., SMB, RPC) from unauthorized sources.                                                     |
| `flag`                    | Connection status flags (S0, REJ, etc.)       | Visitor’s entry behavior (door slammed, door denied)             | Predominantly `SF` (normal establishment and termination) or `S0` (attempt refused).                               | High frequency of `REJ` (rejected) or `RST` (reset) flags indicating failed connection attempts (probing). `S0` without completion (SYN scans). Unusual flag combinations like `FIN` scans for stealthy host discovery.                                                                 |
| `encrypted_service`       | Whether the service typically uses encryption | Visitor uses a secured or private department                     | Increasingly common for privacy (HTTPS, SSH, IMAPS).                                                               | May use encrypted channels (SSH tunnels, HTTPS) to hide malicious command & control (C2) or data exfiltration. May also attack unencrypted services (FTP, Telnet) for eavesdropping.                                                                                                    |
| `encrypted_protocol`      | Encrypted protocol used (HTTPS, SSH)          | Visitor enters through specialized secure corridors              | Legitimate secure access to web servers (HTTPS) or system administration (SSH).                                    | Use of encrypted protocols to obfuscate attack payloads and evade signature-based IDS. Establishment of encrypted tunnels for covert communication.                                                                                                                                     |
| `potential_encrypted`     | Proxy for encrypted traffic                   | Visitor carries items in a locked briefcase                      | May be set for connections to standard secure ports.                                                               | Could be set for malicious traffic masquerading as encrypted service on non-standard ports, or for malware using TLS to communicate.                                                                                                                                                   |
| `encrypted_traffic_score` | Continuous score for likelihood of encryption | Likelihood visitor’s communication is private                    | Higher scores for connections to known secure ports/protocols.                                                     | High scores for traffic that mimics encrypted patterns or uses SSL/TLS on unusual ports, indicating potential C2 traffic. Low scores for attacks targeting plain-text services.                                                                                                        |
| `is_encrypted`            | Binary flag for encrypted traffic             | Visitor is confirmed to use secured rooms                        | Set for confirmed TLS/SSL or SSH sessions.                                                                         | Set for malicious sessions that have successfully established an encrypted channel, making payload inspection difficult.                                                                                                                                                                 |

---

## **3️⃣ Traffic Volume Features**

| Feature           | What It Captures                          | Intuitive Analogy                                               | Normal Behaviour                                                                                                                          | Attacker Behaviour                                                                                                                                                                                                                                                |
| ----------------- | ----------------------------------------- | --------------------------------------------------------------- | ----------------------------------------------------------------------------------------------------------------------------------------- | ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `src_bytes`       | Bytes sent from source to destination     | Amount of documents visitor brings into the building            | Moderate and appropriate to the request (e.g., small HTTP GET, larger FTP upload).                                                        | Can be **extremely small** (e.g., single-packet scans, exploit code) or **extremely large** (e.g., in DDoS attacks where source sends massive floods, or data exfiltration where large files are uploaded from an internal host).                                 |
| `dst_bytes`       | Bytes sent from destination to source     | Amount of information visitor receives back                     | Moderate and appropriate to the response (e.g., webpage content, file download).                                                          | Can be **near zero** (e.g., in a port scan response) or **extremely large** (e.g., in DDoS amplification attacks, or malware downloading a large payload).                                                                                                        |
| `total_bytes`     | Total payload exchanged                   | Total documents exchanged                                       | Balanced or proportional to the service interaction.                                                                                       | Often highly **imbalanced** (e.g., one-way flood in DoS) or **unusually small total** (e.g., single-packet attacks, failed probes).                                                                                                                               |
| `total_bytes_bin` | Quantile-binned total bytes               | Categorizes visitors by small, medium, large document exchanges | Most connections fall into "small" or "medium" bins for typical transactions.                                                              | Often fall into the extreme "small" (scans, exploits) or extreme "large" (floods, exfiltration) bins, making them statistical outliers.                                                                                                                           |
| `byte_ratio`      | Ratio of src/dst bytes                    | Whether visitor brings more than they take back                 | Ratio varies but often has some balance (e.g., ~1 for interactive sessions, >1 for uploads, <1 for downloads).                            | Extremely high (>>1) in cases of data exfiltration/uploads or DoS floods from source. Extremely low (~0) in cases of scanning (no response) or amplification attacks (massive response to small query).                                                           |
| `byte_asymmetry`  | Normalized difference of src vs dst bytes | Visitor’s exchange is lopsided (more in one direction)          | Typically low asymmetry for interactive sessions (e.g., SSH, web browsing).                                                                | High positive asymmetry (source-dominated) in DoS/outbound exfiltration. High negative asymmetry (destination-dominated) in DDoS amplification or inbound malware downloads.                                                                                       |
| `byte_symmetry`   | Indicates balanced traffic flow           | Visitor exchanges similar amounts both ways                     | True for many interactive protocols (e.g., chat, database queries).                                                                       | Often False for attack traffic, which is heavily skewed in one direction (e.g., floods, scans). True for some sophisticated attacks that maintain bidirectional communication (e.g., interactive shell, C2 heartbeat).                                             |

---

## **4️⃣ Error & Reliability Features**

| Feature                   | What It Captures                | Intuitive Analogy                                   | Normal Behaviour                                                                                                               | Attacker Behaviour                                                                                                                                                                                                                                                                        |
| ------------------------- | ------------------------------- | --------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------ | ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `serror_rate`             | Connection/session failure rate | Visitor struggles to open doors                     | Low rate; most connections are successful.                                                                                     | High rate indicating many failed connection attempts, characteristic of **SYN scans, brute-force attacks**, or attempts to connect to closed/unavailable services.                                                                                                                        |
| `srv_serror_rate`         | Service-specific failure rate   | Visitor repeatedly fails at a particular department | Low rate for a specific service.                                                                                               | High rate for a specific service, indicating targeted attacks like **SSH password brute-forcing**, or exploitation attempts against a vulnerable service that is patched/blocked.                                                                                                          |
| `rerror_rate`             | Remote errors                   | Visitor’s requests rejected by staff                | Low rate; remote services typically respond correctly.                                                                         | High rate suggesting the attacker is sending malformed packets or requests that cause the remote service to reply with error codes, or indicative of **reconnaissance probing** that triggers error responses.                                                                            |
| `srv_rerror_rate`         | Service-level remote errors     | Visitor’s department-specific requests rejected     | Low rate for a specific service.                                                                                               | High rate for a specific service, indicating focused attempts to trigger errors (e.g., **buffer overflow probes, SQL injection fuzzing**) that result in remote application errors.                                                                                                        |
| `combined_error_rate`     | Average of all error rates      | Overall visitor struggles summed up                 | Low combined rate, indicating smooth operation.                                                                                | High combined rate is a strong signal of **probing or brute-force activity**, where multiple types of connection failures are occurring.                                                                                                                                                   |
| `dst_host_combined_error` | Average error per host          | Certain departments repeatedly fail visitors        | Low for a host, indicating it's serving requests reliably.                                                                     | High for a host, suggesting it is **under active attack** (e.g., being scanned, brute-forced) or that the attacker is deliberately targeting it with error-inducing packets.                                                                                                               |
| `low_error_rate`          | Flags minimal errors            | Visitor follows rules, no mistakes                  | True for the vast majority of benign, successful connections.                                                                  | Often False for noisy attacks (scans, bruteforce). Can be True for **stealthy, successful attacks** that do not generate protocol errors (e.g., a well-crafted SQL injection that doesn't cause visible errors).                                                                           |
| `clean_fragmentation`     | Packets not fragmented          | Visitor carries complete documents                  | Most modern traffic is not fragmented; fragmentation is handled by networks if needed.                                         | Attackers may use **packet fragmentation** to evade simple IDS/ firewall rules. Therefore, a *lack* of fragmentation (`clean_fragmentation`=True) is typical for normal traffic, while malicious traffic might sometimes have `False`. However, its absence is not a definitive indicator. |

---

## **5️⃣ Host & Service Interaction Features**

| Feature                      | What It Captures                         | Intuitive Analogy                                                                             | Normal Behaviour                                                                                                                                                 | Attacker Behaviour                                                                                                                                                                                                                                                                                             |
| ---------------------------- | ---------------------------------------- | --------------------------------------------------------------------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------------- | -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `count`                      | Total connections to host                | Total number of visitors entering the building                                                | Moderate level, varies with host popularity and time of day.                                                                                                     | **Very High:** In DoS/DDoS attacks, victim host sees a massive surge in connections. **Moderate/Low but Suspicious:** For targeted attacks, count may be low but the connections are malicious.                                                                                                                |
| `srv_count`                  | Connections to specific service          | Visitors heading to one department                                                            | Varies, but a server will see many connections to its primary service (e.g., web server gets many HTTP/S).                                                       | **Very High:** DoS attack focused on a single service (e.g., HTTP flood). **Unusual Pattern:** High `srv_count` for a non-primary/administrative service (e.g., many SMB connections to a web server).                                                                                                         |
| `connection_intensity_ratio` | Focus of traffic on a service            | Most visitors go straight to the main department (high ratio) vs wandering around (low ratio) | High for specialized servers (e.g., a dedicated database server). Lower for multi-service hosts.                                                                  | **Extremely High:** Indicates a concentrated attack on one service (e.g., SYN flood on port 80). **Unusually Low for the Host:** Could indicate horizontal scanning across many services on a single host.                                                                                                     |
| `service_diversity_score`    | How diverse the accessed services are    | Visitor checks many departments vs just one                                                   | For a client, moderate diversity (web, mail, DNS). For a server, low diversity (mainly its advertised service).                                                  | **High for a Client:** Indicates **horizontal scanning** by a single source across multiple ports/services on a target network. **Unexpectedly High for a Server:** Suggests an external host probing all its services.                                                                                         |
| `service_exploration`        | Extent of exploring different services   | Visitor actively poking around multiple departments                                           | Low for normal users who access a predictable set of services.                                                                                                   | High, indicating **reconnaissance activity** where an attacker is mapping available services on a host or network ("service discovery").                                                                                                                                                                       |
| `host_concentration`         | Concentration of traffic on same service | Most visitors concentrated in one department                                                  | High for popular, single-service hosts.                                                                                                                          | Extremely high during a **focused DoS attack** on a specific service. Can also be exploited - an attacker might generate traffic that looks like normal concentration to blend in.                                                                                                                             |
| `host_diversity`             | Spread of traffic across services        | Visitor disperses to multiple departments                                                     | High for multi-purpose servers or user workstations.                                                                                                             | **Low during Vertical Scans:** An attacker focuses many connections on a single service of a single host. **High during Horizontal Scans:** An attacker makes single connections to many different services.                                                                                                   |
| `srv_diff_host_rate`         | Service accessed across multiple hosts   | Visitor tries same department in many buildings                                               | Low for internal clients accessing a common internal service (e.g., domain controller). Can be high for public services (e.g., many hosts querying a public DNS). | **High for an Internal Service:** Could indicate **internal reconnaissance or worm propagation**, where one internal host tries to connect to the same service (e.g., SMB, RPC) on many other internal hosts. **Low for a Public Facing Attack:** A DoS attack might come from many hosts (`count` high) but to a single service on one target (`srv_diff_host_rate` low). |

---

## **6️⃣ User Privilege & Security Features**

| Feature                 | What It Captures                      | Intuitive Analogy                             | Normal Behaviour                                                                                                                                 | Attacker Behaviour                                                                                                                                                                                                                                                                |
| ----------------------- | ------------------------------------- | --------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------ | --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `root_shell`            | Access to root shell                  | Visitor gets master key to building           | Almost never obtained in a standard network connection log. A privileged administrative action.                                                  | If `1`, it is a **major red flag** indicating successful **privilege escalation** or **remote root compromise** (U2R attack). The attacker has gained highest-level control.                                                                                                      |
| `su_attempted`          | Attempted privilege escalation        | Visitor tries switching departments illegally | Very rare in network logs; relates to local system command attempts.                                                                             | If `1`, indicates the attacker, after initial access, is trying to **escalate privileges** on the compromised host (e.g., using `su` or `sudo` commands). Core to U2R attacks.                                                                                                    |
| `num_root`              | Number of root accesses               | Visitor repeatedly enters master rooms        | Should be 0 for virtually all network connections.                                                                                               | >0 indicates **multiple root-level operations** performed in a single connection session. This is a definitive sign of a **successful high-severity compromise** and post-exploitation activity.                                                                                  |
| `num_file_creations`    | Files created                         | Visitor leaves new documents                  | Low (0 or 1) for most sessions (e.g., uploading a file via FTP).                                                                                 | High numbers suggest **post-exploitation activity** like dropping malware, tools, or creating backdoors on the victim system. Part of R2L and U2R attacks.                                                                                                                        |
| `num_shells`            | Shell commands executed               | Visitor performs complex operations           | 0 for almost all application-layer network protocols.                                                                                            | >0 indicates the attacker has **opened a command shell** on the victim machine via the network connection (e.g., via a reverse shell or exploit). A clear sign of successful exploitation and remote control.                                                                     |
| `num_access_files`      | Accessed sensitive files              | Visitor touches restricted documents          | 0 for most connections. Legitimate admin access might be logged differently.                                                                     | >0 suggests the attacker is **accessing or downloading sensitive files** (e.g., /etc/passwd, configuration files). Indicative of **information gathering** or **data theft** (R2L attack).                                                                                        |
| `privilege_indicators`  | Sum of all privilege actions          | Quantifies total elevated access attempts     | Very low (typically 0).                                                                                                                          | A high composite score is a strong, aggregated signal of **post-compromise activity**, combining root access, file ops, and shell activity. Very reliable for detecting U2R attacks.                                                                                              |
| `has_privilege_attempt` | Binary flag for any privilege attempt | Visitor attempts any restricted access        | False for 99.9%+ of connections.                                                                                                                 | True if **any** of the above privilege-related features is triggered. A very efficient filter for high-severity attacks.                                                                                                                                                          |
| `num_failed_logins`     | Failed login attempts                 | Visitor repeatedly fails to enter doors       | Occasional failures are normal (typos). Usually 0 or 1 per session.                                                                              | **High numbers (>>2)** are the hallmark of **brute-force or dictionary attacks** against login services (SSH, FTP, Telnet, web forms). A primary indicator of R2L attacks.                                                                                                        |
| `logged_in`             | Successful login                      | Visitor successfully enters building          | 1 for legitimate authenticated sessions (SSH, FTP, mail, etc.).                                                                                  | 1 can also be the result of a **successful brute-force or credential stuffing attack**, giving the attacker authenticated access. Distinguishing this from normal requires context (source IP, `num_failed_logins` prior).                                                       |
| `is_host_login`         | Host login flag                       | Visitor accesses main building                | Rare; indicates a login to the host itself (e.g., via console, RDP, direct SSH).                                                                 | If set from a network connection, could indicate **administrative access gained by an attacker**, potentially via compromised credentials.                                                                                                                                        |
| `is_guest_login`        | Guest login flag                      | Visitor enters as temporary guest             | Occasionally used for limited access (e.g., FTP guest account).                                                                                  | Attackers may exploit weak default guest accounts to gain initial foothold. Its presence, especially with other suspicious flags, can be indicative of an R2L attempt.                                                                                                            |
| `num_compromised`       | Number of compromised accounts        | Visitor hijacks other visitors’ badges        | Should be 0.                                                                                                                                     | >0 suggests the attacker has **compromised user accounts** on the system, possibly through sniffing or exploitation. Indicates a serious breach and is part of lateral movement or data theft.                                                                                    |
| `num_outbound_cmds`     | Outgoing commands                     | Visitor sends instructions to other rooms     | 0 for most interactive protocols (FTP, HTTP). Present in some like mail (SMTP) or chat.                                                          | In context of an attack, >0 could indicate the compromised host is being used to **send spam, launch further attacks, or perform C2 beaconing** to external servers. Often seen in botnet activity.                                                                               |

---

## **7️⃣ Risk & Attack Features**

| Feature              | What It Captures                                    | Intuitive Analogy                                          | Normal Behaviour                                                                               | Attacker Behaviour                                                                                                                                                                                                                           |
| -------------------- | --------------------------------------------------- | ---------------------------------------------------------- | ---------------------------------------------------------------------------------------------- | -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `attack_risk_score`  | Composite score of protocol, service, errors, flags | Visitor raises multiple red flags at once                  | Low score for connections exhibiting normal, expected behavior.                                | High score indicates a connection exhibits **multiple suspicious characteristics** simultaneously (e.g., wrong protocol for service, high error rates, suspicious flag). A strong aggregated indicator of malice.                            |
| `anomaly_score`      | Continuous measure of unusual activity              | Visitor behaves differently from normal crowd patterns     | Most connections have a low anomaly score, fitting established baselines.                      | High score marks the connection as a **statistical outlier** across many features (duration, bytes, service, etc.). Effective for detecting novel attacks that don't match known signatures but are still abnormal.                          |
| `potential_zero_day` | Flag for unknown/new attacks                        | Visitor does something never seen before                   | Almost always 0.                                                                               | Set to 1 for activity that is highly anomalous, uses unknown exploits, or targets unknown vulnerabilities. Represents the **highest-priority alerts** for novel threats.                                                                     |
| `is_attack`          | Binary attack label                                 | Visitor confirmed malicious                                | 0 for all normal traffic.                                                                      | 1. This is the **ground truth target label** indicating the connection is part of a documented attack (DoS, Probe, R2L, U2R).                                                                                                                 |
| `difficulty_level`   | Detection difficulty                                | How subtle or obvious the visitor’s suspicious behavior is | Not applicable to normal traffic (or considered "very easy" to classify as normal).            | Higher difficulty indicates a **stealthier attack** designed to mimic normal traffic (e.g., low-and-slow scanning, encrypted C2). Lower difficulty indicates **noisy, obvious attacks** (e.g., SYN floods, loud brute-forcing).               |
| `attack_category`    | Type of attack (DoS, Probe, R2L, U2R)              | Visitor’s intent: flood, scan, local attack, root access   | `normal`                                                                                       | **DoS:** Denial of Service - intent to overwhelm. **Probe:** Surveillance/scanning. **R2L:** Unauthorized access from remote (e.g., brute-force). **U2R:** User to Root - privilege escalation on a local host. Categorizes the attack's goal. |

---

## **8️⃣ Behavioral / Burst Features**

| Feature         | What It Captures                                 | Intuitive Analogy                        | Normal Behaviour                                                                      | Attacker Behaviour                                                                                                                                                                                                      |
| --------------- | ------------------------------------------------ | ---------------------------------------- | ------------------------------------------------------------------------------------- | ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `traffic_burst` | Rapid spikes: many connections in short duration | Visitor floods building in short bursts  | Occurs during legitimate events (software updates, conference calls). Usually modest. | A key signature of **DoS/DDoS floods** and **aggressive scanning**, where an attacker maximizes attempts per second to overwhelm targets or complete reconnaissance quickly.                                              |
| `long_duration` | Unusually long connection durations              | Visitor stays in building unusually long | Rare; may occur for large downloads, backups, or long-running sessions (SSH, VPN).    | Indicates **persistent threats**: backdoor shells, C2 channels maintaining connectivity, slow data exfiltration, or **slowloris-type DoS attacks** that hold connections open to exhaust resources.                     |

---

## **9️⃣ Engineered Encryption & Traffic Features**

| Feature                | What It Captures                      | Intuitive Analogy                                   | Normal Behaviour                                                                                         | Attacker Behaviour                                                                                                                                                                                                         |
| ---------------------- | ------------------------------------- | --------------------------------------------------- | -------------------------------------------------------------------------------------------------------- | -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `encrypted_likelihood` | Probability connection is encrypted   | Visitor communicates using locked or private rooms  | High for connections to known secure ports (443, 22). Low for clear-text services (80, 21, 23).          | Attackers may **abuse high-likelihood channels** (HTTPS) to hide malware delivery/C2. They may also **impersonate low-likelihood services** on non-standard ports to evade policy checks.                                   |
| `potential_encrypted`  | Proxy for encrypted traffic           | Visitor carries secured briefcase                   | Set for traffic that has characteristics of encryption (e.g., high entropy payload).                     | Malware using custom encryption or tunneling protocols may trigger this flag, even if not on a standard encrypted port, helping detect **obfuscated C2 traffic**.                                                          |
| `encrypted_service`    | Indicates encryption-prone services   | Visitor goes to secured department                  | True for services like HTTPS, SSH, IMAPS where encryption is standard.                                   | Attackers targeting these services may be trying to **intercept or crack weak encryption**, or using them as a foothold. Lack of encryption (`False`) makes services prime targets for sniffing and credential theft.      |
| `encrypted_protocol`   | Flags encrypted protocols             | Visitor enters through secure corridors             | True for protocols like TLS, SSH, IPSec.                                                                 | Use of encryption itself is not malicious, but its **absence where expected** (e.g., HTTP instead of HTTPS on a login page) is a vulnerability. Attackers also use encrypted protocols to **evade payload inspection** by IDS. |
| `byte_symmetry`        | Balance of traffic in both directions | Visitor exchanges similar amount in both directions | True for interactive applications (SSH, VoIP, database queries).                                         | Often False for attacks (one-way floods, scans). Can be True for sophisticated, **interactive attack sessions** (e.g., an attacker manually operating a remote shell, or C2 beaconing with request/response).              |

---